# コイン参照コードの使用例：コントラクトにapproveする例

## 概要
1. YENコインとEUROコインの２種類のコインをデプロイする
2. ２種類のコインを交換するスマートコントラクトをデプロイする
3. スマートコントラクトを使って、user1とuser2の間でYENコインとEUROコインを交換する

## 準備

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { loadWallet } = await import('../lib/load-wallet.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

動作確認に使用するユーザをロードします。

In [2]:
var users = [];
for(var i=0; i<=2; i++){
    var wfstr = await loadWallet(`wallet-user${i}.json`);
    var wallet = await api.unlockWalletFile(await api.parseWalletFile(wfstr), '_paswrd_');
    var id = (await rpc.call(wallet, 'c1query', { type: 'search', key: 'me' })).value[0].id;
    users[i] = { id, wallet };
    console.log(`user${i}:`, id, wallet.address);
}

user0: u73621973 eQiA3bsfZunQ8KsRpkTgMGndwu46rq
user1: u28169743 epGyEm4BXFxFvLmXkDdDvcqspESPFu
user2: u53371386 eaZMspgRpi68EdReTZEiUkdJuKsdct


## コインのデプロイ

コインコントラクトのサンプルコードを読み出します。

In [3]:
var { default: func } = await import('./coin100.mjs');
var m = /function.*\{([\s\S]*)\}/.exec(func.toString());
var code = m[1];

コインの名称と略号を書き換えます。

In [4]:
var code = code.replace(/COIN_NAME = '.*'/, `COIN_NAME = 'sample YEN COIN'`);
var code = code.replace(/COIN_SYMBOL = '.*'/, `COIN_SYMBOL = 'YEN'`);

YENコインの発行数を10000とし、発行元をユーザ1に書き換えます。  


In [5]:
var code = code.replace(/TOTAL_SUPPLY = \d+/, `TOTAL_SUPPLY = 10000`);
var code = code.replace(/INITIAL_OWNER = '.*'/, `INITIAL_OWNER = '${users[1].id}'`);

スマートコントラクトをデプロイします。  
変数yen_cidにコインコントラクトのIDが格納されます。

In [6]:
var func = new Function(code);
var yen_cid = await deploySmartContract({func:'string', args:'any'}, func, { name: 'yen-coin' });

次にEUROコインをデプロイします。 コインの名称と略号を書き換えます。

In [7]:
var code = code.replace(/COIN_NAME = '.*'/, `COIN_NAME = 'sample EURO COIN'`);
var code = code.replace(/COIN_SYMBOL = '.*'/, `COIN_SYMBOL = 'EURO'`);

EUROコインの発行数を100とし、発行元をユーザ2に書き換えます。  


In [8]:
var code = code.replace(/TOTAL_SUPPLY = \d+/, `TOTAL_SUPPLY = 100`);
var code = code.replace(/INITIAL_OWNER = '.*'/, `INITIAL_OWNER = '${users[2].id}'`);

スマートコントラクトをデプロイします。  
変数euro_cidにコインコントラクトのIDが格納されます。

In [9]:
var func = new Function(code);
var euro_cid = await deploySmartContract({func:'string', args:'any'}, func, { name: 'euro-coin' });

コインコントラクトのアクセス範囲を、動作確認に使用するユーザに設定します。

In [10]:
await rpc.call(adminWallet, 'c1update', {id:yen_cid, prop:'accessible_to', value: users.map(e => e.id)});
await rpc.call(adminWallet, 'c1update', {id:euro_cid, prop:'accessible_to', value: users.map(e => e.id)});

{
  txno: 702121,
  txid: 'xVC9BLQSTnexing9zTN4PPx7yaW9X6gwhfeF3hPSp5fEGB',
  status: 'ok',
  value: null
}

## swapコントラクトをデプロイ

本例題用のコントラクトをデプロイします。  
ユーザ１とユーザ２の間でYENコインとEUROコインを交換する処理を行うコントラクトです。  
交換のレートはEURO/YEN=150です。  
これをswapコントラクトと呼ぶことにします。  
ユーティリティdeploySmartContractを使ってデプロイします。  
変数swap_cidにコントラクトのIDが返ります。

In [11]:
var swap_cid = await deploySmartContract({n: 'number'}, function coin_swap(n) {
    openContract('__yen__').call({func: 'transferFrom', args: { from: '__user1__', to: '__user2__', value: 150*n }});
    openContract('__euro__').call({func: 'transferFrom', args: { from: '__user2__', to: '__user1__', value: n }});
}, {replacers: [
    [/__yen__/g, yen_cid],
    [/__euro__/g, euro_cid],
    [/__user1__/g, users[1].id],
    [/__user2__/g, users[2].id]
]});

作成したswapコントラクトがコインコントラクトにアクセスできるように、accessible_toを設定します。

In [12]:
await rpc.call(adminWallet, 'c1update', {id:yen_cid, prop:'add accessible_to', value: [swap_cid]});
await rpc.call(adminWallet, 'c1update', {id:euro_cid, prop:'add accessible_to', value: [swap_cid]});

{
  txno: 702126,
  txid: 'xN9nbrAKN4Mb9EQXZBy3fxiKY6vLGu9ywpby9v86EDTHr',
  status: 'ok',
  value: null
}

作成したコントラクトの内容を確認します。

In [13]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'a_contract', id: swap_cid });
console.log(resp.value);

{
  id: [ 'c019597231', 'coin_swap@c.TDSL.H011' ],
  frozen: null,
  name: 'coin_swap',
  domain: [ 'd93319890', '@c.TDSL.H011' ],
  description: '',
  argtypes: { n: 'number' },
  code: "    openContract('c044999308').call({func: 'transferFrom', args: { from: 'u28169743', to: 'u53371386', value: 150*n }});    openContract('c043207521').call({func: 'transferFrom', args: { from: 'u53371386', to: 'u28169743', value: n }});",
  mask: {},
  maxsteps: 0,
  a_txno: 702122,
  c_txno: 702122,
  num_transactions: 0,
  last_active: 1753421480296,
  created_at: 1753421480296,
  accessible_to: [ [ 'u58281888', 'jupyter@c.TDSL.H011' ] ],
  callable_to: [ 'all' ],
  disclosed_to: [],
  groups: [],
  managed: true,
  accessible: true
}


## 実行例１

いきなりswapコントラクトを実行すると、エラーとなります。  
なぜなら、swapコントラクトにはコインを転送する権限が無いからです。

In [14]:
await rpc.call(adminWallet, swap_cid, { n: 1 });

{
  txno: 702127,
  txid: 'xVyJQ52xYorMo9zNPFwr6qSUCJH4LEnZcpeRowbuFMQQ7',
  status: 'aborted',
  value: 'thrown: insufficient allowance\n    at c019597231:1:5'
}

当然、コインの保持数は初期状態のままです。

In [15]:
var resp1 = await rpc.call(users[0].wallet, yen_cid, {func: 'balanceOf', args: {owner: users[1].id}});
var resp2 = await rpc.call(users[0].wallet, euro_cid, {func: 'balanceOf', args: {owner: users[1].id}});
console.log('balanceOf(user1):', `${resp1.value}YEN`,`${resp2.value}EURO`);
var resp1 = await rpc.call(users[0].wallet, yen_cid, {func: 'balanceOf', args: {owner: users[2].id}});
var resp2 = await rpc.call(users[0].wallet, euro_cid, {func: 'balanceOf', args: {owner: users[2].id}});
console.log('balanceOf(user2):', `${resp1.value}YEN`,`${resp2.value}EURO`);

balanceOf(user1): 10000YEN 0EURO
balanceOf(user2): 0YEN 100EURO


## 実行例２

user1の1000YENコインの転送を、swapコントラクトにapproveします。

In [16]:
var resp = await rpc.call(users[1].wallet, yen_cid, {func: 'approve', args: {spender: swap_cid, value: 1000}});
console.log(resp);

{
  txno: 702133,
  txid: 'xqhYKrfZT3B4yFevpqXiKsLPtyaJzozBrT2jnarRGMvQBB',
  status: 'ok',
  value: true
}


念のため、allowanceで確認します。

In [17]:
var resp = await rpc.call(users[1].wallet, yen_cid, {func: 'allowance', args: {owner: users[1].id, spender: swap_cid }});
console.log(resp);

{
  txno: 702134,
  txid: 'xGhXNzrZUXvModcauMoZfFVyrp5XEZmZVmH9Gm7RXiCkDB',
  status: 'ok',
  value: 1000
}


この状態で、swapコントラクトを実行すると、依然としてエラーとなります。  
なぜなら、YENコインは転送できても、swapコントラクトにEUROコインを転送する権限が無いからです。  
エラーの起きる場所が前と変わっていることがわかります。

In [18]:
var resp = await rpc.call(adminWallet, swap_cid, { n: 1 });console.log(resp);

{
  txno: 702135,
  txid: 'xdm6PeZnBCQ4UPtNQdavmn7VuQZ8nDcVGDXz3AckxUx8GB',
  status: 'aborted',
  value: 'thrown: insufficient allowance\n    at c019597231:1:125'
}


この場合でも、コインの保持数が変わることはありません。  
トランザクションがエラーになることで、すべての変更が巻き戻されるからです。（先に行われたYENコインの転送は元にもどります）

In [19]:
var resp1 = await rpc.call(users[0].wallet, yen_cid, {func: 'balanceOf', args: {owner: users[1].id}});
var resp2 = await rpc.call(users[0].wallet, euro_cid, {func: 'balanceOf', args: {owner: users[1].id}});
console.log('balanceOf(user1):', `${resp1.value}YEN`,`${resp2.value}EURO`);
var resp1 = await rpc.call(users[0].wallet, yen_cid, {func: 'balanceOf', args: {owner: users[2].id}});
var resp2 = await rpc.call(users[0].wallet, euro_cid, {func: 'balanceOf', args: {owner: users[2].id}});
console.log('balanceOf(user2):', `${resp1.value}YEN`,`${resp2.value}EURO`);


balanceOf(user1): 10000YEN 0EURO
balanceOf(user2): 0YEN 100EURO


## 実行例３

さらに、user2の10EUROコインの転送を、swapコントラクトにapproveします。

In [20]:
var resp = await rpc.call(users[2].wallet, euro_cid, {func: 'approve', args: {spender: swap_cid, value: 10}});
console.log(resp);

{
  txno: 702142,
  txid: 'xa6ufJbwYeUvdgArtLH3EUme5rbJEdvZ5ugVLs8cTKpZRB',
  status: 'ok',
  value: true
}


念のため、allowanceで確認します。

In [21]:
var resp = await rpc.call(users[2].wallet, euro_cid, {func: 'allowance', args: {owner: users[2].id, spender: swap_cid }});
console.log(resp);

{
  txno: 702143,
  txid: 'xBV5qD2fN3jk4Npyap4dJcBp82nc998K2voZFWpyhkbfFB',
  status: 'ok',
  value: 10
}


この状態で、swapコントラクトを実行すると、実行は成功します。

In [22]:
var resp = await rpc.call(adminWallet, swap_cid, { n: 1 });
console.log(resp);

{
  txno: 702144,
  txid: 'x95paT37UA94DtFcxX3MEJASmTBy6qjsF7AzbfNzszyDKB',
  status: 'ok',
  value: null
}


コインが交換されたことを確認します。

In [23]:
var resp1 = await rpc.call(users[0].wallet, yen_cid, {func: 'balanceOf', args: {owner: users[1].id}});
var resp2 = await rpc.call(users[0].wallet, euro_cid, {func: 'balanceOf', args: {owner: users[1].id}});
console.log('balanceOf(user1):', `${resp1.value}YEN`,`${resp2.value}EURO`);
var resp1 = await rpc.call(users[0].wallet, yen_cid, {func: 'balanceOf', args: {owner: users[2].id}});
var resp2 = await rpc.call(users[0].wallet, euro_cid, {func: 'balanceOf', args: {owner: users[2].id}});
console.log('balanceOf(user2):', `${resp1.value}YEN`,`${resp2.value}EURO`);


balanceOf(user1): 9850YEN 1EURO
balanceOf(user2): 150YEN 99EURO


念のため、allowanceが減っていることを確認します。

In [24]:
var resp = await rpc.call(users[1].wallet, yen_cid, {func: 'allowance', args: {owner: users[1].id, spender: swap_cid }});
console.log('allowance YEN', resp.value);var resp = await rpc.call(users[2].wallet, euro_cid, {func: 'allowance', args: {owner: users[2].id, spender: swap_cid }});
console.log('allowance EURO', resp.value);

allowance YEN 850
allowance EURO 9


## 実行例４

さらに交換を行います。

In [25]:
var resp = await rpc.call(adminWallet, swap_cid, { n: 1 });
console.log(resp);

{
  txno: 702153,
  txid: 'xRFbh7TGaFK6meQbbT2jyBSB2N7pdq642XkT2yqYTfV48',
  status: 'ok',
  value: null
}


コインがさらに交換されたことを確認します。

In [26]:
var resp1 = await rpc.call(users[0].wallet, yen_cid, {func: 'balanceOf', args: {owner: users[1].id}});
var resp2 = await rpc.call(users[0].wallet, euro_cid, {func: 'balanceOf', args: {owner: users[1].id}});
console.log('balanceOf(user1):', `${resp1.value}YEN`,`${resp2.value}EURO`);
var resp1 = await rpc.call(users[0].wallet, yen_cid, {func: 'balanceOf', args: {owner: users[2].id}});
var resp2 = await rpc.call(users[0].wallet, euro_cid, {func: 'balanceOf', args: {owner: users[2].id}});
console.log('balanceOf(user2):', `${resp1.value}YEN`,`${resp2.value}EURO`);


balanceOf(user1): 9700YEN 2EURO
balanceOf(user2): 300YEN 98EURO


念のため、allowanceがさらに減っていることを確認します。

In [27]:
var resp = await rpc.call(users[1].wallet, yen_cid, {func: 'allowance', args: {owner: users[1].id, spender: swap_cid }});
console.log('allowance YEN', resp.value);var resp = await rpc.call(users[2].wallet, euro_cid, {func: 'allowance', args: {owner: users[2].id, spender: swap_cid }});
console.log('allowance EURO', resp.value);

allowance YEN 700
allowance EURO 8


## 実行例５

さらに交換を繰り返します。allowanceが不足するまで繰り返します。

In [28]:
while(true){    var resp = await rpc.call(adminWallet, swap_cid, { n: 1 });
    console.log(resp);    if(resp.status !== 'ok') break;}

{
  txno: 702162,
  txid: 'xKG7nSLTLrMbGt7D7mBhXVBGS6LBW5rYTVBGzCXZhu4pBB',
  status: 'ok',
  value: null
}
{
  txno: 702165,
  txid: 'x2nVb4cyii9aSxxckWWJWoTnRxrGKJm4RH3pspp2RsjxLB',
  status: 'ok',
  value: null
}
{
  txno: 702168,
  txid: 'xJQcf3va3L9826GQRnesuF8VKHjZYesXY8vaui9AuPkzJB',
  status: 'ok',
  value: null
}
{
  txno: 702171,
  txid: 'x8rrg7nGMxmcNLgvHuGQBsBDCgnwjKRCKMiiRdxvbUvfz',
  status: 'ok',
  value: null
}
{
  txno: 702174,
  txid: 'xjcpbSNpSRxWbEaCETqs4TA89cjNLEKLJKGqYhEUVDdNKB',
  status: 'aborted',
  value: 'thrown: insufficient allowance\n    at c019597231:1:5'
}


コインが過不足なく交換されたことを確認します。

In [29]:
var resp1 = await rpc.call(users[0].wallet, yen_cid, {func: 'balanceOf', args: {owner: users[1].id}});
var resp2 = await rpc.call(users[0].wallet, euro_cid, {func: 'balanceOf', args: {owner: users[1].id}});
console.log('balanceOf(user1):', `${resp1.value}YEN`,`${resp2.value}EURO`);
var resp1 = await rpc.call(users[0].wallet, yen_cid, {func: 'balanceOf', args: {owner: users[2].id}});
var resp2 = await rpc.call(users[0].wallet, euro_cid, {func: 'balanceOf', args: {owner: users[2].id}});
console.log('balanceOf(user2):', `${resp1.value}YEN`,`${resp2.value}EURO`);


balanceOf(user1): 9100YEN 6EURO
balanceOf(user2): 900YEN 94EURO


念のため、allowanceを確認します。

In [30]:
var resp = await rpc.call(users[1].wallet, yen_cid, {func: 'allowance', args: {owner: users[1].id, spender: swap_cid }});
console.log('allowance YEN', resp.value);var resp = await rpc.call(users[2].wallet, euro_cid, {func: 'allowance', args: {owner: users[2].id, spender: swap_cid }});
console.log('allowance EURO', resp.value);

allowance YEN 100
allowance EURO 4


このような仕組みで、アトミックにコインの交換を実現することができました。